# 03-9 문법 종합 실습: 이벤트 검토 큐

03-1~03-8의 변수·자료형, 컬렉션, 조건문, 반복문, 함수, 예외, 클래스를 하나의 메모리 기반 프로그램으로 연결합니다. 이 노트북은 데이터·저장소·명령·세션 로직을 한 커널에서 확인하는 풀이 검증용이며 패키지 분리, 상대 import, 공개 API, `python -m` 실행은 다루지 않습니다. 해당 항목은 `examples/03-9-event-review-starter`에서 직접 완성하고 검증하세요. 시작 코드를 먼저 구현한 뒤 필요할 때만 이 노트북과 비교하며, 각 코드 셀은 위에서 아래로 순서대로 실행합니다. 파일·네트워크·외부 패키지는 사용하지 않습니다.

## 1. 이벤트 데이터클래스와 경계값 검증

먼저 포트 변환 규칙과 이벤트 객체를 정의합니다. 기본 실습에서는 일반 `@dataclass`와 `__post_init__()` 대입으로 정규화 흐름을 드러냅니다. `bool`은 `int`의 하위 타입이므로 `type(value)`를 명시적으로 검사합니다.

In [ ]:
from dataclasses import dataclass, field


def expect_exception(expected_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_type as exc:
        return str(exc)
    except Exception as exc:
        raise AssertionError(
            f"{expected_type.__name__} 대신 {type(exc).__name__} 발생"
        )
    raise AssertionError(f"{expected_type.__name__}이 발생하지 않았습니다")


def parse_port(value):
    if type(value) not in {int, str}:
        raise TypeError("port는 정수 또는 정수 문자열이어야 합니다")
    try:
        port = int(value)
    except ValueError:
        raise ValueError("port를 정수로 변환할 수 없습니다")
    if not 1 <= port <= 65535:
        raise ValueError("port는 1~65535 범위여야 합니다")
    return port


@dataclass
class SecurityEvent:
    action: str
    ip: str
    port: int

    def __post_init__(self):
        if not isinstance(self.action, str):
            raise TypeError("action은 문자열이어야 합니다")
        self.action = self.action.strip().upper()
        if self.action not in {"ALLOW", "DENY"}:
            raise ValueError("action은 ALLOW 또는 DENY여야 합니다")
        if not isinstance(self.ip, str):
            raise TypeError("ip는 문자열이어야 합니다")
        self.ip = self.ip.strip()
        if not self.ip:
            raise ValueError("ip는 비어 있을 수 없습니다")
        self.port = parse_port(self.port)

    def endpoint(self):
        return f"{self.ip}:{self.port}"


event = SecurityEvent(" allow ", " 192.0.2.10 ", "443")
assert event == SecurityEvent("ALLOW", "192.0.2.10", 443)
assert event.endpoint() == "192.0.2.10:443"
print(event)

In [ ]:
assert "ALLOW 또는 DENY" in expect_exception(
    ValueError, SecurityEvent, "BLOCK", "192.0.2.10", 443
)
assert "비어" in expect_exception(
    ValueError, SecurityEvent, "ALLOW", "", 443
)
assert "1~65535" in expect_exception(
    ValueError, SecurityEvent, "ALLOW", "192.0.2.10", 0
)
assert "정수 또는 정수 문자열" in expect_exception(
    TypeError, SecurityEvent, "ALLOW", "192.0.2.10", True
)
print("이벤트 경계값 검증 통과")

## 2. 상태를 책임지는 저장소

이벤트 목록은 `EventStore`가 소유합니다. 외부에는 튜플 스냅샷을 반환하고, 인스턴스별 기본 목록은 `default_factory`로 생성합니다.

In [ ]:
@dataclass
class EventStore:
    _events: list[SecurityEvent] = field(default_factory=list)

    def add(self, event):
        if not isinstance(event, SecurityEvent):
            raise TypeError("SecurityEvent만 추가할 수 있습니다")
        if event in self._events:
            raise ValueError("동일한 이벤트가 이미 있습니다")
        self._events.append(event)

    def list_all(self):
        return tuple(self._events)

    def find_by_action(self, action):
        if not isinstance(action, str):
            raise TypeError("action은 문자열이어야 합니다")
        normalized = action.strip().upper()
        if normalized not in {"ALLOW", "DENY"}:
            raise ValueError("action은 ALLOW 또는 DENY여야 합니다")
        matches = []
        for event in self._events:
            if event.action == normalized:
                matches.append(event)
        return tuple(matches)

    def remove(self, number):
        if type(number) is not int:
            raise TypeError("삭제 번호는 정수여야 합니다")
        if not 1 <= number <= len(self._events):
            raise IndexError("삭제 번호가 목록 범위를 벗어났습니다")
        return self._events.pop(number - 1)

    def summary(self):
        counts = {"total": len(self._events), "ALLOW": 0, "DENY": 0}
        for event in self._events:
            counts[event.action] += 1
        return counts


store_check = EventStore()
allow_event = SecurityEvent("ALLOW", "192.0.2.10", 443)
deny_event = SecurityEvent("DENY", "198.51.100.4", 22)
store_check.add(allow_event)
store_check.add(deny_event)
assert store_check.list_all() == (allow_event, deny_event)
assert store_check.find_by_action("allow") == (allow_event,)
assert store_check.summary() == {"total": 2, "ALLOW": 1, "DENY": 1}
assert store_check.remove(2) == deny_event
print(store_check.summary())

In [ ]:
first_store = EventStore()
second_store = EventStore()
first_store.add(SecurityEvent("ALLOW", "203.0.113.1", 80))
assert second_store.list_all() == ()
snapshot = first_store.list_all()
assert isinstance(snapshot, tuple)
assert "이미" in expect_exception(
    ValueError, first_store.add, SecurityEvent("ALLOW", "203.0.113.1", 80)
)
print("저장소 독립성·중복 방지·읽기 전용 스냅샷 검증 통과")

## 3. 명령 파싱과 실행

문자열 파싱, 도메인 상태 변경, 출력 형식을 분리하면 각 책임을 독립적으로 검증할 수 있습니다.

In [ ]:
@dataclass
class Command:
    name: str
    arguments: tuple[str, ...] = ()


COMMAND_ARGUMENT_COUNTS = {
    "add": 3,
    "list": 0,
    "find": 1,
    "summary": 0,
    "remove": 1,
    "quit": 0,
}


def parse_command(text):
    if not isinstance(text, str):
        raise TypeError("명령은 문자열이어야 합니다")
    parts = text.split()
    if not parts:
        raise ValueError("명령이 비어 있습니다")
    name = parts[0].lower()
    arguments = tuple(parts[1:])
    if name not in COMMAND_ARGUMENT_COUNTS:
        raise ValueError(f"지원하지 않는 명령: {name}")
    expected = COMMAND_ARGUMENT_COUNTS[name]
    if len(arguments) != expected:
        raise ValueError(
            f"{name} 명령은 인자 {expected}개가 필요합니다: 현재 {len(arguments)}개"
        )
    return Command(name=name, arguments=arguments)


assert parse_command(" ADD allow 192.0.2.10 443 ") == Command(
    "add", ("allow", "192.0.2.10", "443")
)
assert parse_command("summary") == Command("summary")
assert "지원하지 않는" in expect_exception(ValueError, parse_command, "unknown")
print("명령 파서 검증 통과")

In [ ]:
def format_event(number, event):
    return f"{number}. {event.action} {event.ip} {event.port}"


def format_events(events):
    if not events:
        return ["목록이 비어 있습니다"]
    messages = []
    for number, event in enumerate(events, start=1):
        messages.append(format_event(number, event))
    return messages


def format_matches(events):
    if not events:
        return ["검색 결과가 없습니다"]
    messages = []
    for event in events:
        messages.append(f"{event.action} {event.ip} {event.port}")
    return messages


def format_summary(summary):
    return (
        f"total={summary['total']} "
        f"ALLOW={summary['ALLOW']} "
        f"DENY={summary['DENY']}"
    )


def execute_command(store, command):
    if not isinstance(store, EventStore):
        raise TypeError("store는 EventStore여야 합니다")
    if not isinstance(command, Command):
        raise TypeError("command는 Command여야 합니다")
    if command.name == "add":
        action, ip, port = command.arguments
        event = SecurityEvent(action, ip, port)
        store.add(event)
        return True, [f"추가: {event.action} {event.endpoint()}"]
    if command.name == "list":
        return True, format_events(store.list_all())
    if command.name == "find":
        (action,) = command.arguments
        return True, format_matches(store.find_by_action(action))
    if command.name == "summary":
        return True, [format_summary(store.summary())]
    if command.name == "remove":
        (number_text,) = command.arguments
        try:
            number = int(number_text)
        except ValueError:
            raise ValueError("삭제 번호를 정수로 변환할 수 없습니다")
        removed = store.remove(number)
        return True, [f"삭제: {removed.action} {removed.endpoint()}"]
    if command.name == "quit":
        return False, ["종료합니다"]
    raise RuntimeError(f"처리되지 않은 명령: {command.name}")

## 4. 오류 복구가 가능한 세션

사용자 입력 오류는 메시지로 바꾸고 다음 명령을 계속 처리합니다. `quit`은 정상적인 종료 신호입니다.

In [ ]:
def process_input(store, text):
    command = parse_command(text)
    return execute_command(store, command)


def run_session(commands, store=None):
    if store is None:
        store = EventStore()
    outputs = []
    for text in commands:
        try:
            keep_running, messages = process_input(store, text)
        except (ValueError, IndexError) as exc:
            outputs.append(f"오류: {exc}")
            continue
        outputs.extend(messages)
        if not keep_running:
            break
    return store, outputs

In [ ]:
commands = [
    "list",
    "add allow 192.0.2.10 443",
    "add DENY 198.51.100.4 22",
    "add ALLOW 192.0.2.10 443",
    "add BLOCK 203.0.113.8 80",
    "add ALLOW 203.0.113.8 https",
    "list",
    "find allow",
    "summary",
    "remove 2",
    "summary",
    "unknown",
    "quit",
    "add DENY 203.0.113.9 53",
]

store, outputs = run_session(commands)
assert store.summary() == {"total": 1, "ALLOW": 1, "DENY": 0}
assert outputs[0] == "목록이 비어 있습니다"
assert outputs[6:8] == [
    "1. ALLOW 192.0.2.10 443",
    "2. DENY 198.51.100.4 22",
]
assert outputs[8] == "ALLOW 192.0.2.10 443"
assert outputs[-1] == "종료합니다"
error_count = 0
for output in outputs:
    if output.startswith("오류:"):
        error_count += 1
assert error_count == 4
for output in outputs:
    print(output)

In [ ]:
error_commands = [
    "",
    "add ALLOW 192.0.2.1",
    "find BLOCK",
    "remove one",
    "remove 1",
    "quit now",
    "quit",
]
error_store, error_outputs = run_session(error_commands)
assert error_store.list_all() == ()
assert len(error_outputs) == 7
for line in error_outputs[:-1]:
    assert line.startswith("오류:")
assert error_outputs[-1] == "종료합니다"
print("오류 경로 6종과 정상 종료 검증 통과")

## 5. 터미널 경계와 자동 시나리오

`main()`은 실제 `input()`과 `print()`을 가장 바깥에서만 연결합니다. 노트북에서 `main()`을 직접 호출하면 사용자 입력을 기다리므로, 같은 핵심 함수를 사용하는 `run_session()`으로 자동 시나리오를 검증합니다.

In [ ]:
def main():
    store = EventStore()
    print("명령: add/list/find/summary/remove/quit")
    while True:
        text = input("event> ")
        try:
            keep_running, messages = process_input(store, text)
        except (ValueError, IndexError) as exc:
            print(f"오류: {exc}")
            continue
        for message in messages:
            print(message)
        if not keep_running:
            return 0


terminal_commands = [
    "add ALLOW 192.0.2.10 443",
    "summary",
    "quit",
]
terminal_store, terminal_outputs = run_session(terminal_commands)
assert terminal_store.summary() == {"total": 1, "ALLOW": 1, "DENY": 0}
assert "total=1 ALLOW=1 DENY=0" in terminal_outputs
assert terminal_outputs[-1] == "종료합니다"
for output in terminal_outputs:
    print(output)

## 6. 완료 점검과 확장 방향

완료 기준: 정상 명령 6종, 오류 후 계속 실행, 중복 방지, 범위 검사, 인스턴스별 상태 분리, `quit` 이후 미실행, 터미널과 분리된 자동 시나리오를 확인했습니다. `find`는 삭제 번호와 혼동하지 않게 결과를 번호 없이 보여 주고, `remove`는 `list`에 나타난 번호만 사용합니다. 다음에는 기능을 한 번에 늘리지 말고 `find-port PORT`처럼 하나의 명령을 추가한 뒤 파서 → 저장소 → 실행기 → 검증 순서로 확장해 보세요. 파일 저장과 불러오기는 04장에서 다룹니다.

In [ ]:
final_checks = {
    "이벤트 데이터 정규화": event.endpoint() == "192.0.2.10:443",
    "저장소 캡슐화": isinstance(snapshot, tuple),
    "대표 시나리오": outputs[-1] == "종료합니다",
    "오류 복구": len(error_outputs) == 7,
    "터미널 밖 자동 시나리오": terminal_outputs[-1] == "종료합니다",
}
for name, passed in final_checks.items():
    assert passed
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")